# Assignment 22: Embedding Models, Vector Stores & Similarity Search

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# PART 1 - Embedding Models

## Task 1: OpenAI Embedding Model

In [30]:
from langchain_community.document_loaders import TextLoader,PyMuPDFLoader
from langchain_text_splitters import CharacterTextSplitter,RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings

In [31]:
loader = PyMuPDFLoader('../Data/Sample-filled-in-MR.pdf')
docs = loader.load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = splitter.split_documents(docs)

In [32]:
len(chunks)

21

In [33]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [34]:
chunks

[Document(metadata={'producer': '', 'creator': '', 'creationdate': '2015-09-19T08:45:51+08:00', 'source': '../Data/Sample-filled-in-MR.pdf', 'file_path': '../Data/Sample-filled-in-MR.pdf', 'total_pages': 10, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2015-09-19T08:45:51+08:00', 'trapped': '', 'modDate': "D:20150919084551+08'00'", 'creationDate': "D:20150919084551+08'00'", 'page': 0}, page_content='- 1 - \n \nSAMPLE \n(All names and details provided in this sample are fictitious.  \nSome fields have been deliberately left blank.) \n \n \nMEDICAL REPORT \n \nSECTION 1: PATIENT’S PARTICULARS \n \nFull name of patient: Mr Tan Ah Kow \n \nNRIC/FIN/Passport no. of patient: S1111111X \n \nAge of patient: 55 years old \n \nSECTION 2: DOCTOR’S PARTICULARS \n \nFull name of doctor: Tan Ah Moi \n \nNRIC/FIN/Passport no. of doctor: S2222222Z  \n \n \nMCR no. of doctor: 333333 \n \nHospital / Clinic name and address: 1 Blackacre Hospital, Singapore 01

In [35]:
texts = [chunk.page_content for chunk in chunks]

In [36]:
vectors = embeddings.embed_documents(texts)

In [38]:
len(vectors)

21

In [39]:
print(vectors)

[[0.00356292724609375, -0.0040130615234375, 0.048370361328125, 0.043212890625, 0.0082550048828125, -0.03277587890625, -0.0122222900390625, -0.01214599609375, -0.017547607421875, 0.007015228271484375, 0.01007080078125, -0.044342041015625, 0.024993896484375, 0.00982666015625, 0.054351806640625, 0.0325927734375, 0.026824951171875, 0.0251312255859375, 0.01274871826171875, -0.01206207275390625, 0.033111572265625, 0.051116943359375, 0.0276947021484375, -0.025634765625, -0.01163482666015625, -0.05120849609375, 0.00711822509765625, -0.005290985107421875, 0.027923583984375, -0.0279541015625, -0.003429412841796875, -0.0204925537109375, 0.0083770751953125, 0.032745361328125, -0.027374267578125, 0.0204315185546875, -0.007602691650390625, -0.01216888427734375, 0.010162353515625, -0.036956787109375, -0.0128936767578125, -0.0182037353515625, 0.00146484375, 0.0178070068359375, 0.0362548828125, -0.005218505859375, 0.01111602783203125, -0.0172882080078125, 0.033599853515625, 0.041595458984375, -0.023757

## Task 2: Hugging Face Embedding Model

In [40]:
from langchain_huggingface import HuggingFaceEmbeddings

In [41]:
embedding = HuggingFaceEmbeddings(model='sentence-transformers/all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [43]:
texts = [chunk.page_content for chunk in chunks]

In [44]:
doc_vector = embedding.embed_documents(texts)

In [47]:
print(f'Hugging Face Vectors :{doc_vector}\n OpenAI Vectors : {vectors}')

Hugging Face Vectors :[[-0.05156921222805977, 0.12136361747980118, -0.019058747217059135, 0.03180494159460068, -0.03638697788119316, 0.04459920525550842, 0.03926287218928337, 0.0390608049929142, -0.033230338245630264, -0.048162560909986496, -0.0072738598100841045, -0.11871758103370667, -0.012364593334496021, 0.03221575543284416, -0.05009571462869644, -0.06268743425607681, -0.030230296775698662, -0.023356636986136436, 0.03038886934518814, -0.06350679695606232, -0.017551686614751816, 0.0728839859366417, 0.03742067888379097, -0.09760906547307968, -0.04589409381151199, -0.03342615067958832, 0.0036591317038983107, -0.008824331685900688, 0.00029262786847539246, 0.01952236518263817, 0.046238645911216736, 0.07144670188426971, 0.03465619310736656, 0.005691517144441605, 0.05873823165893555, -0.029304765164852142, -0.08272597193717957, 0.09276239573955536, -0.026058271527290344, 0.04822612553834915, -0.004434924107044935, -0.053617849946022034, 0.06122890114784241, -0.01028106827288866, 0.0485199

## Task 3: Comparison : OpenAI vs Hugging Face Embeddings

# PART 2 : Document Similarity Search (Open AI Embeddings)

## Task 4: Similarity Search App (core logic)

In [55]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma.vectorstores import Chroma

In [56]:
texts

['- 1 - \n \nSAMPLE \n(All names and details provided in this sample are fictitious.  \nSome fields have been deliberately left blank.) \n \n \nMEDICAL REPORT \n \nSECTION 1: PATIENT’S PARTICULARS \n \nFull name of patient: Mr Tan Ah Kow \n \nNRIC/FIN/Passport no. of patient: S1111111X \n \nAge of patient: 55 years old \n \nSECTION 2: DOCTOR’S PARTICULARS \n \nFull name of doctor: Tan Ah Moi \n \nNRIC/FIN/Passport no. of doctor: S2222222Z  \n \n \nMCR no. of doctor: 333333 \n \nHospital / Clinic name and address: 1 Blackacre Hospital, Singapore 01010101 \n \n \n \nDoctor’s qualifications and experience in this area of work: \n \n \n[To set out details]',
 '- 2 - \n \n \nDoctor-patient relationship: \n \nPlease state if you have been seeing the patient regularly over a period of time (if so, \nplease state when you first started seeing the patient and how often you see the \npatient) or if you saw the patient specifically for this mental capacity assessment only. \n \n \nI have been the

In [50]:
openai_embeddings = OpenAIEmbeddings()

In [51]:
vector_store = Chroma.from_documents(chunks,openai_embeddings)

In [52]:
vector_store_with_embeding = Chroma.from_documents(chunks,embeddings)

In [53]:
vector_store

In [54]:
vector_store_with_embeding

In [57]:
def Search(query,k=3):
    searches = vector_store_with_embeding.similarity_search(query,k=k)
    return searches

In [61]:
Q1 = Search('What is the Patient Name ?')

In [62]:
Q1

[Document(id='c3abb801-58e2-436d-98b3-ddeefebdec37', metadata={'format': 'PDF 1.5', 'subject': '', 'producer': '', 'creationdate': '2015-09-19T08:45:51+08:00', 'keywords': '', 'moddate': '2015-09-19T08:45:51+08:00', 'creationDate': "D:20150919084551+08'00'", 'total_pages': 10, 'title': '', 'file_path': '../Data/Sample-filled-in-MR.pdf', 'creator': '', 'page': 0, 'trapped': '', 'modDate': "D:20150919084551+08'00'", 'author': '', 'source': '../Data/Sample-filled-in-MR.pdf'}, page_content='- 1 - \n \nSAMPLE \n(All names and details provided in this sample are fictitious.  \nSome fields have been deliberately left blank.) \n \n \nMEDICAL REPORT \n \nSECTION 1: PATIENT’S PARTICULARS \n \nFull name of patient: Mr Tan Ah Kow \n \nNRIC/FIN/Passport no. of patient: S1111111X \n \nAge of patient: 55 years old \n \nSECTION 2: DOCTOR’S PARTICULARS \n \nFull name of doctor: Tan Ah Moi \n \nNRIC/FIN/Passport no. of doctor: S2222222Z  \n \n \nMCR no. of doctor: 333333 \n \nHospital / Clinic name and 

In [63]:
Q2 = Search("What is the Patient Diagnosis ?")

In [64]:
Q2

[Document(id='e6603294-455f-4f40-a314-910f665c0d35', metadata={'creator': '', 'page': 4, 'trapped': '', 'file_path': '../Data/Sample-filled-in-MR.pdf', 'author': '', 'source': '../Data/Sample-filled-in-MR.pdf', 'keywords': '', 'subject': '', 'modDate': "D:20150919084551+08'00'", 'format': 'PDF 1.5', 'title': '', 'total_pages': 10, 'creationdate': '2015-09-19T08:45:51+08:00', 'moddate': '2015-09-19T08:45:51+08:00', 'creationDate': "D:20150919084551+08'00'", 'producer': ''}, page_content='- 5 - \n \nDiagnosis: \n1. Dementia  \n2. Stroke \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \n \nSECTION 4: OPINION ON PATIENT’S MENTAL CAPACITY \n \nOPINION ON PATIENT’S MENTAL CAPACITY IN RELATION TO \nPERSONAL WELFARE \n \nIn your opinion, can the patient understand information relevant to a decision \nrelating to his or her personal welfare? \n \n\uf0a8 Yes \n\uf0fe No \n \nIn your opinion, can the patient retain information long enough to make a \ndecision relating to his or her personal welfa